# Первый поиск похожих случаев: TF-IDF

В этом ноутбуке одна строка датасета считается одним клиническим случаем. По тексту `description` найдём другие случаи с похожими словами и короткими фразами.

Для первого baseline используем слова и биграммы без лемматизации и удаления стоп-слов. TF-IDF строит векторы описаний, а косинусное сходство ранжирует кандидатов. Это лексический поиск: он учитывает совпадение текстовых признаков, но не понимает синонимы как языковая модель.

`bethesda` оставляем только для последующей оценки выдачи. Поиск не предсказывает категорию Bethesda. Ячейка загрузки находит корень проекта по `requirements.txt`, поэтому путь к данным не зависит от того, открыто ядро из корня проекта, папки ноутбука или `.venv`.

## 1. Загрузка корпуса

Сначала поднимаемся от текущей рабочей папки до корня проекта, распознавая его по `requirements.txt`. Затем читаем идентификатор случая, описание и категорию Bethesda. В признаки поиска ниже попадёт только `description`; категория нужна, чтобы после ранжирования сравнить результаты с категорией запроса.

In [2]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "requirements.txt").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError(
            "Не удалось найти корень проекта с requirements.txt."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_first.csv"
cases = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=["case_id", "description", "bethesda"],
)

## 2. Проверка корпуса

Проверяем только размер таблицы и число пустых описаний. Сами тексты здесь не выводим.

In [3]:
print(f"Строк в таблице: {len(cases)}")
print(f"Пустых description: {cases['description'].isna().sum()}")

Строк в таблице: 5587
Пустых description: 0


## 3. Подготовка текстов

Убираем пропуски и описания, состоящие только из пробелов. Остальные тексты пока не лемматизируем и не фильтруем по стоп-словам: это сохраняет простой baseline.

In [4]:
has_description = cases["description"].notna()
has_description &= cases["description"].str.strip().ne("")
search_cases = cases.loc[has_description].copy().reset_index(drop=True)

if search_cases.empty:
    raise ValueError("В корпусе нет непустых описаний для поиска.")

print(f"Описаний для поиска: {len(search_cases)}")

Описаний для поиска: 5587


## 4. Построение TF-IDF-векторов

`TfidfVectorizer` создаёт признаки из отдельных слов (униграмм) и пар соседних слов (биграмм), а затем взвешивает их по TF-IDF. `fit_transform` одновременно строит словарь по корпусу и превращает все описания в разреженную матрицу: строки — случаи, столбцы — текстовые признаки.

По умолчанию векторизатор приводит текст к нижнему регистру и разбивает его на токены. Односимвольные токены он не учитывает; это ограничение базовой настройки, которое можно отдельно проверить позже.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(search_cases["description"])

print(f"Размер матрицы: {tfidf_matrix.shape[0]} случаев × {tfidf_matrix.shape[1]} признаков")

Размер матрицы: 5587 случаев × 29610 признаков


## 5. Выбор описания-запроса

Для первого примера берём описание из самого корпуса. `query_position` — его позиция в `search_cases`, начиная с нуля. Поменяй число, чтобы выбрать другой случай. Позже вместо этой строки можно будет подставить новый текст.

In [11]:
query_position = 5
query_case = search_cases.iloc[query_position]
query_text = query_case["description"]
query_vector = vectorizer.transform([query_text])

print(f"Ищем случаи, похожие на case_id={query_case['case_id']}")
print(query_text)

Ищем случаи, похожие на case_id=T1_000028
П 5 (№1179/23). На фоне элементов периферической крови обнаружены группы клеток активно пролиферирующего фолликулярного эпителия аденоматозного зоба с формированием вакуолей резорбции коллоида и нескольких микрофолликулярных структур, позволяющих рекомендовать динамическое наблюдение.


## 6. Ранжирование похожих случаев

Сравниваем вектор запроса со всеми строками корпуса косинусным сходством. Чем выше оценка, тем больше совпадение TF-IDF-признаков. Строку запроса исключаем, чтобы случай не оказался самым похожим на самого себя.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(query_vector, tfidf_matrix).ravel()
similarities[query_position] = -1.0

TOP_K = 10
ranked_positions = similarities.argsort()[::-1]
ranked_positions = [
    position
    for position in ranked_positions
    if position != query_position
][:TOP_K]

results = search_cases.iloc[ranked_positions][
    ["case_id", "description", "bethesda"]
].copy()
results.insert(1, "similarity", similarities[ranked_positions])
results = results.reset_index(drop=True)

display(results)

,case_id,similarity,description,bethesda
0,T1_019431,0.575187,П 22 (№6869/22). На фоне элементов периферичес...,II
1,T1_000078,0.559753,Л 15 (№1195/23). На фоне элементов периферичес...,II
2,T2_000466,0.548971,"П 2 (№2594/23 1,2 - 2 мазка). В двух мазках на...",II
3,T1_004346,0.546336,"Л 3 (№334/23). На фоне ""жидкого"" коллоида, изм...",II
4,T1_021964,0.523926,Л 9 (№8556/22). На фоне элементов периферическ...,II
5,T2_000624,0.513468,П 6 (№2396/23). На фоне элементов периферическ...,II
6,T1_023022,0.505987,П 5 (№316/23). На фоне элементов периферическо...,II
7,T1_023563,0.502415,"П 8 (№881/23 1,2 - 2 мазка). На фоне ""жидкого""...",II
8,T1_019720,0.499624,П 1 (№7034/22). На фоне элементов периферическ...,II
9,T1_007523,0.480218,"Л 2 (№299/22 1,2 - 2 мазка). В двух мазках на ...",IV


## 7. Проверка по категории Bethesda

Пока считаем совпадение категории Bethesda с категорией запроса приближённым сигналом релевантности. Это не эталон клинической похожести и не часть поисковой модели: сравниваем категории только после ранжирования.

In [13]:
query_bethesda = query_case["bethesda"]

if pd.isna(query_bethesda):
    print("У запроса не указана категория Bethesda; оценить совпадение категорий нельзя.")
else:
    same_bethesda = results["bethesda"].eq(query_bethesda)
    results_with_evaluation = results.assign(
        same_bethesda_as_query=same_bethesda
    )
    display(
        results_with_evaluation[
            ["case_id", "similarity", "bethesda", "same_bethesda_as_query"]
        ]
    )
    print(
        "Доля top-"
        f"{len(results)} случаев с той же Bethesda: "
        f"{same_bethesda.mean():.2f}"
    )

,case_id,similarity,bethesda,same_bethesda_as_query
0,T1_019431,0.575187,II,True
1,T1_000078,0.559753,II,True
2,T2_000466,0.548971,II,True
3,T1_004346,0.546336,II,True
4,T1_021964,0.523926,II,True
5,T2_000624,0.513468,II,True
6,T1_023022,0.505987,II,True
7,T1_023563,0.502415,II,True
8,T1_019720,0.499624,II,True
9,T1_007523,0.480218,IV,False


Доля top-10 случаев с той же Bethesda: 0.90


## Как интерпретировать результат

- `similarity` — сходство текстовых TF-IDF-векторов, а не вероятность того, что случаи клинически эквивалентны.
- `same_bethesda_as_query` показывает только совпадение категории с запросом. Совпавшая категория не гарантирует похожесть описаний.
- Просмотри тексты нескольких верхних результатов и отметь, согласуются ли они с твоей интуицией. При оценке помни: словесное совпадение может пропустить синонимы и перефразировки.
- Для следующего сравнения можно изменить один фактор, например добавить лемматизацию, и повторить поиск с тем же запросом.